# Cyclone Amphan – Downloading Data To Google Drive


| Folder | Dataset | Official source |
|---|---|---|
| `1_IFS_HRES` | IFS HRES 0.25°, init 13–15 May 2020 (00/12 UTC), leads 3–10 d | WeatherBench 2 (Google Cloud Zarr) |
| `2_ERA5_Climatology` | ERA5 6-hourly, 1–31 May, 1991–2020 (+ mean/std) | WeatherBench 2 (Google Cloud Zarr) |
| `3_ERA5_Surface` | ERA5 hourly: May/Nov 2015–2019 (train), 13–25 May 2020 (test) | Copernicus CDS API |
| `4_IBTrACS` | IBTrACS v04r01 (North Indian basin) + Amphan track | NOAA NCEI |
| `5_IMERG` | IMERG Final 0.1° V07, 13–25 May 2020 | NASA GES DISC via `earthaccess` |

In [ ]:
!pip -q install xarray zarr gcsfs netcdf4 h5netcdf dask cdsapi earthaccess tqdm

## Cell 1 – Mount Drive, folders and region setting

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os

ROOT = '/content/drive/MyDrive/Amphan_Data'
DIRS = {
    'hres':      f'{ROOT}/1_IFS_HRES',
    'era5_clim': f'{ROOT}/2_ERA5_Climatology',
    'era5_sfc':  f'{ROOT}/3_ERA5_Surface',
    'ibtracs':   f'{ROOT}/4_IBTrACS',
    'imerg':     f'{ROOT}/5_IMERG',
}
for p in DIRS.values():
    os.makedirs(p, exist_ok=True)

# Spatial window applied to IFS HRES, ERA5 climatology and ERA5 surface downloads.
# Global 0.25 deg ERA5 for all years would be 100+ GB, so the default is the North Indian Ocean.
# Set REGION = None for the full globe (much bigger and slower).
REGION = dict(lat=(0, 35), lon=(60, 110))   # degrees N, degrees E (0-360)

def subset(ds):
    if REGION is None:
        return ds
    lat0, lat1 = REGION['lat']; lon0, lon1 = REGION['lon']
    asc = float(ds.latitude[0]) < float(ds.latitude[-1])
    ds = ds.sel(latitude=slice(lat0, lat1) if asc else slice(lat1, lat0))
    return ds.sel(longitude=slice(lon0, lon1))

print('Saving to:', ROOT)

Mounted at /content/drive
Saving to: /content/drive/MyDrive/Amphan_Data


## Cell 2 – Dataset 1: IFS HRES 0.25°
Variables `msl`, `u10`, `v10` (WB2 names: `mean_sea_level_pressure`, `10m_u_component_of_wind`, `10m_v_component_of_wind`). Init 13–15 May 2020 at 00/12 UTC, lead 3–10 days. One NetCDF per init time.

In [ ]:
import gcsfs, xarray as xr, numpy as np, pandas as pd, os

fs = gcsfs.GCSFileSystem(token='anon')

def pick_zarr(folder, must):
    c = [p.rstrip('/') for p in fs.ls(f'weatherbench2/datasets/{folder}')
         if p.rstrip('/').endswith('.zarr') and all(m in p for m in must)]
    if not c:
        print('Available:', fs.ls(f'weatherbench2/datasets/{folder}'))
        raise FileNotFoundError(f'No zarr in {folder} matching {must}')
    return 'gs://' + sorted(c, key=lambda p: ('derived' in p, len(p)))[0]

hres_path = pick_zarr('hres', ['1440x721'])          # 0.25 deg, 00/12 UTC inits
print('Using:', hres_path)
ds = xr.open_zarr(hres_path, chunks={}, storage_options={'token': 'anon'})

VARS = ['mean_sea_level_pressure', '10m_u_component_of_wind', '10m_v_component_of_wind']
ds = subset(ds[VARS])
inits = pd.to_datetime([f'2020-05-{d} {h}:00' for d in (13, 14, 15) for h in ('00', '12')])
leads = slice(np.timedelta64(3, 'D'), np.timedelta64(10, 'D'))
enc = {v: {'zlib': True, 'complevel': 4} for v in VARS}

for t in inits:
    out = f"{DIRS['hres']}/ifs_hres_0p25_init_{t:%Y%m%dT%H}.nc"
    if os.path.exists(out):
        print('skip', out); continue
    d = ds.sel(time=t, prediction_timedelta=leads).load()
    d.to_netcdf(out + '.part', encoding=enc)
    os.replace(out + '.part', out)          # only becomes a real file once fully written
    print('saved', out, dict(d.sizes))

Using: gs://weatherbench2/datasets/hres/2016-2022-0012-1440x721.zarr


/tmp/ipykernel_2142/2425645634.py:15: FutureWarning: In a future version, xarray will not decode the variable 'prediction_timedelta' into a timedelta64 dtype based on the presence of a timedelta-like 'units' attribute by default. Instead it will rely on the presence of a timedelta64 'dtype' attribute, which is now xarray's default way of encoding timedelta64 values.
To continue decoding into a timedelta64 dtype, either set `decode_timedelta=True` when opening this dataset, or add the attribute `dtype='timedelta64[ns]'` to this variable on disk.
To opt-in to future behavior, set `decode_timedelta=False`.
  ds = xr.open_zarr(hres_path, chunks={}, storage_options={'token': 'anon'})


saved /content/drive/MyDrive/Amphan_Data/1_IFS_HRES/ifs_hres_0p25_init_20200513T00.nc {'prediction_timedelta': 29, 'latitude': 141, 'longitude': 201}
saved /content/drive/MyDrive/Amphan_Data/1_IFS_HRES/ifs_hres_0p25_init_20200513T12.nc {'prediction_timedelta': 29, 'latitude': 141, 'longitude': 201}
saved /content/drive/MyDrive/Amphan_Data/1_IFS_HRES/ifs_hres_0p25_init_20200514T00.nc {'prediction_timedelta': 29, 'latitude': 141, 'longitude': 201}
saved /content/drive/MyDrive/Amphan_Data/1_IFS_HRES/ifs_hres_0p25_init_20200514T12.nc {'prediction_timedelta': 29, 'latitude': 141, 'longitude': 201}
saved /content/drive/MyDrive/Amphan_Data/1_IFS_HRES/ifs_hres_0p25_init_20200515T00.nc {'prediction_timedelta': 29, 'latitude': 141, 'longitude': 201}
saved /content/drive/MyDrive/Amphan_Data/1_IFS_HRES/ifs_hres_0p25_init_20200515T12.nc {'prediction_timedelta': 29, 'latitude': 141, 'longitude': 201}


## Cell 3 – Dataset 2: ERA5 6-hourly May climatology 1991–2020
Downloads every May (00/06/12/18 UTC) as one file per year, then computes the per-grid-node mean and standard deviation for each day-of-May and hour.

In [ ]:
era5_path = pick_zarr('era5', ['6h', '1440x721'])
print('Using:', era5_path)
era5 = xr.open_zarr(era5_path, chunks={}, storage_options={'token': 'anon'})
era5 = subset(era5[VARS])

from tqdm.auto import tqdm
for y in tqdm(range(1991, 2021)):
    out = f"{DIRS['era5_clim']}/era5_6h_may_{y}.nc"
    if os.path.exists(out):
        continue
    d = era5.sel(time=slice(f'{y}-05-01 00:00', f'{y}-05-31 18:00')).load()
    d.to_netcdf(out + '.part', encoding=enc)
    os.replace(out + '.part', out)

# Climatology (mean / std per grid node, per day-of-May and synoptic hour, 30 years)
import glob
files = sorted(glob.glob(f"{DIRS['era5_clim']}/era5_6h_may_*.nc"))
allm = xr.open_mfdataset(files, combine='by_coords').load()
key = allm.time.dt.strftime('%m-%d %H').rename('mmdd_hh')
clim = xr.concat([allm.groupby(key).mean('time'), allm.groupby(key).std('time', ddof=1)],
                 dim=pd.Index(['mean', 'std'], name='stat'))
cpath = f"{DIRS['era5_clim']}/era5_may_climatology_1991_2020_mean_std.nc"
clim.to_netcdf(cpath + '.part'); os.replace(cpath + '.part', cpath)
print(len(files), 'yearly files;', 'climatology saved', dict(clim.sizes))

Using: gs://weatherbench2/datasets/era5/1959-2022-6h-1440x721.zarr


  0%|          | 0/30 [00:00<?, ?it/s]

30 yearly files; climatology saved {'stat': 2, 'mmdd_hh': 124, 'latitude': 141, 'longitude': 201}


## Cell 4 – Copernicus CDS login


In [ ]:
from getpass import getpass
key = getpass('Enter your CDS API key: ')
with open('/root/.cdsapirc', 'w') as f:
    f.write('url: https://cds.climate.copernicus.eu/api\nkey: ' + key.strip() + '\n')
import cdsapi
cdsapi.Client()   # quick credential check
print('CDS credentials OK')

f0994e9d-db4c-4b07-9097-7bd53242d91b··········
CDS credentials OK


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os

ROOT = '/content/drive/MyDrive/Amphan_Data'
DIRS = {
    'hres':      f'{ROOT}/1_IFS_HRES',
    'era5_clim': f'{ROOT}/2_ERA5_Climatology',
    'era5_sfc':  f'{ROOT}/3_ERA5_Surface',
    'ibtracs':   f'{ROOT}/4_IBTrACS',
    'imerg':     f'{ROOT}/5_IMERG',
}
for p in DIRS.values():
    os.makedirs(p, exist_ok=True)

REGION = dict(lat=(0, 35), lon=(60, 110))   # set to None for global

def subset(ds):
    if REGION is None:
        return ds
    lat0, lat1 = REGION['lat']; lon0, lon1 = REGION['lon']
    asc = float(ds.latitude[0]) < float(ds.latitude[-1])
    ds = ds.sel(latitude=slice(lat0, lat1) if asc else slice(lat1, lat0))
    return ds.sel(longitude=slice(lon0, lon1))

print('Saving to:', ROOT)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Saving to: /content/drive/MyDrive/Amphan_Data


## Cell 5 – Dataset 3: ERA5 hourly surface (CDS API)
Variables: `tp, t2m, d2m, msl, u10, v10, sp, vimd`. One request per month (May and Nov 2015–2019) plus 13–25 May 2020 for testing. Requests run 4 at a time.

In [ ]:
import xarray as xr, os, time

ARCO = 'gs://gcp-public-data-arco-era5/ar/full_37-1h-0p25deg-chunk-1.zarr-v3'
ds = xr.open_zarr(ARCO, chunks={'time': 24}, storage_options=dict(token='anon'))

WANT = {   # short name : possible names in the ARCO store
    'tp':   ['total_precipitation'],
    't2m':  ['2m_temperature'],
    'd2m':  ['2m_dewpoint_temperature'],
    'msl':  ['mean_sea_level_pressure'],
    'u10':  ['10m_u_component_of_wind'],
    'v10':  ['10m_v_component_of_wind'],
    'sp':   ['surface_pressure'],
    'vimd': ['vertically_integrated_moisture_divergence',
             'vertical_integral_of_divergence_of_moisture_flux'],
}
use = {}
for k, cands in WANT.items():
    hit = [c for c in cands if c in ds.data_vars]
    if hit:
        use[k] = hit[0]
    else:
        print(f'MISSING in ARCO: {k}  (get this one from CDS later)')
print('Using:', use)

sub = xr.Dataset({k: ds[v] for k, v in use.items()})
enc = {k: {'zlib': True, 'complevel': 4} for k in use}

periods = []
for y in range(2015, 2020):                                   # training
    for m, last in ((5, 31), (11, 30)):
        periods.append((f'train_{y}_{m:02d}', f'{y}-{m:02d}-01 00:00', f'{y}-{m:02d}-{last} 23:00'))
periods.append(('test_2020_05_13-25', '2020-05-13 00:00', '2020-05-25 23:00'))   # Amphan holdout

for name, t0, t1 in periods:
    out = f"{DIRS['era5_sfc']}/era5_arco_sfc_{name}.nc"
    if os.path.exists(out):
        print('skip', name); continue
    t = time.time()
    d = subset(sub.sel(time=slice(t0, t1))).load()
    d.to_netcdf(out + '.part', encoding=enc)
    os.replace(out + '.part', out)                            # only a real file once fully written
    print(f'saved {name}  {dict(d.sizes)}  {(time.time() - t) / 60:.1f} min')

Using: {'tp': 'total_precipitation', 't2m': '2m_temperature', 'd2m': '2m_dewpoint_temperature', 'msl': 'mean_sea_level_pressure', 'u10': '10m_u_component_of_wind', 'v10': '10m_v_component_of_wind', 'sp': 'surface_pressure', 'vimd': 'vertically_integrated_moisture_divergence'}
saved train_2015_05  {'latitude': 141, 'longitude': 201, 'time': 744}  2.2 min
saved train_2015_11  {'latitude': 141, 'longitude': 201, 'time': 720}  2.1 min
saved train_2016_05  {'latitude': 141, 'longitude': 201, 'time': 744}  2.2 min
saved train_2016_11  {'latitude': 141, 'longitude': 201, 'time': 720}  2.1 min
saved train_2017_05  {'latitude': 141, 'longitude': 201, 'time': 744}  2.2 min
saved train_2017_11  {'latitude': 141, 'longitude': 201, 'time': 720}  2.1 min
saved train_2018_05  {'latitude': 141, 'longitude': 201, 'time': 744}  2.2 min
saved train_2018_11  {'latitude': 141, 'longitude': 201, 'time': 720}  2.2 min
saved train_2019_05  {'latitude': 141, 'longitude': 201, 'time': 744}  2.2 min
saved train_

## Cell 6 – Dataset 4: IBTrACS v04r01 (NOAA NCEI)
Downloads the North Indian basin CSV and extracts Cyclone Amphan (2020) with `Timestamp, LAT, LON, WMO_WIND, WMO_PRES`.

In [ ]:
import requests, pandas as pd

base = 'https://www.ncei.noaa.gov/data/international-best-track-archive-for-climate-stewardship-ibtracs/v04r01/access/csv/'
fname = 'ibtracs.NI.list.v04r01.csv'
dest = f"{DIRS['ibtracs']}/{fname}"

r = requests.get(base + fname, stream=True, timeout=120)
r.raise_for_status()
with open(dest + '.part', 'wb') as f:
    for chunk in r.iter_content(1 << 20):
        f.write(chunk)
os.replace(dest + '.part', dest)

df = pd.read_csv(dest, skiprows=[1], low_memory=False)      # row 2 in the file is a units row
am = df[(df['NAME'].str.strip() == 'AMPHAN') & (df['SEASON'].astype(str) == '2020')].copy()
cols = ['SID', 'SEASON', 'NAME', 'ISO_TIME', 'NATURE', 'LAT', 'LON', 'WMO_WIND', 'WMO_PRES']
am = am[cols]
for c in ['LAT', 'LON', 'WMO_WIND', 'WMO_PRES']:
    am[c] = pd.to_numeric(am[c], errors='coerce')
am.to_csv(f"{DIRS['ibtracs']}/amphan_2020_ibtracs.csv", index=False)
print(len(am), 'Amphan records,', am['ISO_TIME'].iloc[0], '->', am['ISO_TIME'].iloc[-1])
am.head()

51 Amphan records, 2020-05-15 06:00:00 -> 2020-05-21 12:00:00


,SID,SEASON,NAME,ISO_TIME,NATURE,LAT,LON,WMO_WIND,WMO_PRES
60427,2020136N10088,2020,AMPHAN,2020-05-15 06:00:00,DS,9.5,87.5,NaN,NaN
60428,2020136N10088,2020,AMPHAN,2020-05-15 09:00:00,DS,9.5,87.2,NaN,NaN
60429,2020136N10088,2020,AMPHAN,2020-05-15 12:00:00,TS,9.5,87.0,NaN,NaN
60430,2020136N10088,2020,AMPHAN,2020-05-15 15:00:00,TS,9.5,86.8,NaN,NaN
60431,2020136N10088,2020,AMPHAN,2020-05-15 18:00:00,TS,9.5,86.6,NaN,NaN


## Cell 7 – NASA Earthdata login

In [ ]:
import earthaccess
auth = earthaccess.login(strategy='interactive', persist=True)
print('Authenticated:', auth.authenticated)

Enter your Earthdata Login username: midushi
Enter your Earthdata password: ··········
Authenticated: True


## Cell 8 – Dataset 5: IMERG Final 0.1° V07, 13–25 May 2020 (GES DISC)
Half-hourly `GPM_3IMERGHH` V07

In [ ]:
results = earthaccess.search_data(
    short_name='GPM_3IMERGHH', version='07',
    temporal=('2020-05-13 00:00:00', '2020-05-25 23:59:59'))
print(len(results), 'granules found (expected 624)')

files = earthaccess.download(results, DIRS['imerg'], threads=8)
print(len(files), 'files in', DIRS['imerg'])

624 granules found (expected 624)


QUEUEING TASKS | :   0%|          | 0/624 [00:00<?, ?it/s]

PROCESSING TASKS | :   0%|          | 0/624 [00:00<?, ?it/s]

COLLECTING RESULTS | :   0%|          | 0/624 [00:00<?, ?it/s]

624 files in /content/drive/MyDrive/Amphan_Data/5_IMERG


## Cell 9 – Merging IMERG into one small regional NetCDF


In [ ]:
import glob
paths = sorted(glob.glob(f"{DIRS['imerg']}/*3IMERG*"))
def load(f):
    d = xr.open_dataset(f, group='Grid')
    v = 'precipitation' if 'precipitation' in d else 'precipitationCal'
    d = d[[v]].rename({v: 'precipitation'})
    if REGION is not None:
        d = d.sel(lat=slice(*REGION['lat']), lon=slice(*REGION['lon']))
    return d.load()

parts = [load(f) for f in paths]
imerg = xr.concat(parts, dim='time')
ipath = f"{DIRS['imerg']}/imerg_final_v07_precip_20200513-25_region.nc"
imerg.to_netcdf(ipath + '.part', encoding={'precipitation': {'zlib': True, 'complevel': 4}})
os.replace(ipath + '.part', ipath)
print(dict(imerg.sizes))

{'time': 624, 'lon': 500, 'lat': 350}


## Cell 10 – To  Check that everything is in Drive


In [ ]:
for k, p in DIRS.items():
    n = sum(len(fs_) for _, _, fs_ in os.walk(p))
    size = sum(os.path.getsize(os.path.join(dp, f)) for dp, _, fs_ in os.walk(p) for f in fs_) / 1e9
    print(f'{os.path.basename(p):22s} {n:4d} files  {size:7.2f} GB')

1_IFS_HRES                6 files     0.03 GB
2_ERA5_Climatology       31 files     0.96 GB
3_ERA5_Surface           11 files     4.10 GB
4_IBTrACS                 2 files     0.03 GB
5_IMERG                 625 files     5.46 GB
